# F4 primer: just enough PyTorch to read Karpathy's GPT code

Throwaway learning code. Run the cells top to bottom (kernel: project `.venv`).

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)  # same "random" numbers every run, so output is reproducible


def show(name, t):
    print(f"{name}  shape={tuple(t.shape)}\n{t}\n")

## 1. Tensors and `(B, T)` shapes
Tensors are numpy arrays that can live on a GPU and track gradients.

In [2]:
B, T = 2, 4  # 2 snippets in a batch, each 4 tokens long
idx = torch.tensor([[5, 2, 7, 1],  # snippet 0: token ids
                    [3, 3, 0, 6]])  # snippet 1
show("idx (token ids)", idx)  # shape (B, T): this is what the data loader yields

idx (token ids)  shape=(2, 4)
tensor([[5, 2, 7, 1],
        [3, 3, 0, 6]])



## 2. `nn.Embedding`: id → vector (adds the C dimension)
A learnable lookup table.

In [3]:
vocab_size, C = 8, 3  # 8 possible tokens, each becomes a 3-number vector
emb = nn.Embedding(vocab_size, C)  # a (vocab_size, C) table of random numbers...
show("embedding table (learned during training)", emb.weight.data)

embedding table (learned during training)  shape=(8, 3)
tensor([[ 1.9269,  1.4873,  0.9007],
        [-2.1055,  0.6784, -1.2345],
        [-0.0431, -1.6047,  0.3559],
        [-0.6866, -0.4934,  0.2415],
        [-1.1109,  0.0915, -2.3169],
        [-0.2168, -0.3097, -0.3957],
        [ 0.8034, -0.6216, -0.5920],
        [-0.0631, -0.8286,  0.3309]])



In [4]:
x = emb(idx)  # ...and calling it just looks up rows
show("x = emb(idx)   <- (B, T) became (B, T, C)", x.data)
print("token 3 appears twice in snippet 1 (same id -> same vector):")
print(x[1, 0].data, x[1, 1].data)

x = emb(idx)   <- (B, T) became (B, T, C)  shape=(2, 4, 3)
tensor([[[-0.2168, -0.3097, -0.3957],
         [-0.0431, -1.6047,  0.3559],
         [-0.0631, -0.8286,  0.3309],
         [-2.1055,  0.6784, -1.2345]],

        [[-0.6866, -0.4934,  0.2415],
         [-0.6866, -0.4934,  0.2415],
         [ 1.9269,  1.4873,  0.9007],
         [ 0.8034, -0.6216, -0.5920]]])

token 3 appears twice in snippet 1 (same id -> same vector):
tensor([-0.6866, -0.4934,  0.2415]) tensor([-0.6866, -0.4934,  0.2415])


## 3. `nn.Linear`: `y = x @ W.T + b`, applied to the last dim

In [5]:
head_size = 2
proj = nn.Linear(C, head_size, bias=False)  # W is (head_size, C) = learned
y = proj(x)
show("y = proj(x)   <- (B, T, C) became (B, T, head_size)", y.data)

y = proj(x)   <- (B, T, C) became (B, T, head_size)  shape=(2, 4, 2)
tensor([[[-0.0813, -0.1152],
         [ 0.3310, -0.6476],
         [ 0.2003, -0.3926],
         [-0.7328, -0.5473]],

        [[ 0.0294, -0.5795],
         [ 0.0294, -0.5795],
         [ 0.2849,  1.2623],
         [ 0.0781,  0.3561]]])



This is exactly what Karpathy's key/query/value layers are: three `nn.Linear(C, head_size)` → three different learned views of each token.

## 4. The trick: each token = average of itself + earlier tokens
Three ways to compute the same thing.

In [6]:
xb = x[0].data  # take snippet 0 only, shape (T, C)

# Version 1: obvious but slow python loop
v1 = torch.stack([xb[: t + 1].mean(dim=0) for t in range(T)])
show("v1 (loop)", v1)

v1 (loop)  shape=(4, 3)
tensor([[-0.2168, -0.3097, -0.3957],
        [-0.1299, -0.9572, -0.0199],
        [-0.1076, -0.9143,  0.0970],
        [-0.6071, -0.5161, -0.2359]])



In [7]:
# Version 2: lower-triangular matrix of weights, one matrix multiply
tril = torch.tril(torch.ones(T, T))
show("tril (1 = 'may look at')", tril)
wei = tril / tril.sum(dim=1, keepdim=True)
show("wei = each row sums to 1 -> 'how much to take from each token'", wei)
v2 = wei @ xb  # (T, T) @ (T, C) -> (T, C)
show("v2 (matmul)", v2)

tril (1 = 'may look at')  shape=(4, 4)
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

wei = each row sums to 1 -> 'how much to take from each token'  shape=(4, 4)
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])

v2 (matmul)  shape=(4, 3)
tensor([[-0.2168, -0.3097, -0.3957],
        [-0.1299, -0.9572, -0.0199],
        [-0.1076, -0.9143,  0.0970],
        [-0.6071, -0.5161, -0.2359]])



In [8]:
# Version 3: same thing written with softmax -- this is the form attention uses
scores = torch.zeros(T, T)  # attention will put REAL q.k scores here instead of 0
scores = scores.masked_fill(tril == 0, float("-inf"))  # future = -inf
show("scores after mask (-inf = cannot see the future)", scores)
wei3 = F.softmax(scores, dim=-1)  # exp(-inf)=0, so the future gets weight 0
v3 = wei3 @ xb
show("v3 (softmax)", v3)

scores after mask (-inf = cannot see the future)  shape=(4, 4)
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

v3 (softmax)  shape=(4, 3)
tensor([[-0.2168, -0.3097, -0.3957],
        [-0.1299, -0.9572, -0.0199],
        [-0.1076, -0.9143,  0.0970],
        [-0.6071, -0.5161, -0.2359]])



In [9]:
print("all three versions equal?", torch.allclose(v1, v2), torch.allclose(v1, v3))

all three versions equal? True True


**The key idea:** in v3 the scores are all zeros → plain average. Self-attention replaces those zeros with q·k scores, so each token chooses *which* past tokens to listen to. That swap is your `Head.forward()` in F4 step 3.